# V6 Model Pipeline — Weather + Stock Features + Bias Correction

**Полный пайплайн**: загрузка Excel -> препроцессинг -> погода -> тюнинг LightGBM (GPU)

### Инструкция для Kaggle:
1. Загрузи `3_month_data.xlsx` как Dataset
2. Включи GPU Accelerator в Settings
3. Run All

In [ ]:
!pip install -q openmeteo-requests requests-cache retry-requests lightgbm optuna openpyxl

In [ ]:
import pandas as pd
import numpy as np
import optuna
import warnings
import openmeteo_requests
import requests_cache
from retry_requests import retry
from lightgbm import LGBMRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Проверяем доступность GPU для LightGBM
try:
    test_m = LGBMRegressor(device='gpu', n_estimators=1, verbose=-1)
    test_m.fit([[1],[2],[3]], [1,2,3])
    USE_GPU = True
    print('GPU доступен для LightGBM')
except Exception:
    USE_GPU = False
    print('GPU недоступен, используем CPU')

## 1. Загрузка и препроцессинг данных

In [ ]:
# === НАСТРОЙ ПУТЬ К ФАЙЛУ ===
# Kaggle: /kaggle/input/<dataset-name>/3_month_data.xlsx
# Локально: data/raw/3_month_data.xlsx

import os

# Автоопределение пути
if os.path.exists('/kaggle/input'):
    # Ищем файл в Kaggle datasets
    for root, dirs, files in os.walk('/kaggle/input'):
        for f in files:
            if f == '3_month_data.xlsx':
                FILE_PATH = os.path.join(root, f)
                break
    print(f'Kaggle: {FILE_PATH}')
else:
    FILE_PATH = 'data/raw/3_month_data.xlsx'
    print(f'Local: {FILE_PATH}')

In [ ]:
def load_raw_data(file_path: str) -> pd.DataFrame:
    """Умная загрузка: определяет нужен ли сдвиг заголовка."""
    probe = pd.read_excel(file_path, header=None, nrows=12)
    row10_values = probe.iloc[10].astype(str).tolist()
    if any('Дата' in v for v in row10_values if isinstance(v, str)):
        print('  Обнаружена служебная шапка (10 строк)')
        df = pd.read_excel(file_path, header=10)
        df = df.dropna(axis=1, how='all')
    else:
        df = pd.read_excel(file_path)
    return df


def preprocess_data(file_path: str) -> pd.DataFrame:
    """Полный препроцессинг: аномалии, лаги, stock-фичи, календарь."""
    print(f'Загрузка данных из: {file_path}...')
    df = load_raw_data(file_path)
    print(f'Загружено строк: {len(df):,}')

    # 1. Обработка аномалий
    for col in ['Продано', 'Остаток', 'Выпуск']:
        neg = (df[col] < 0).sum()
        df[col] = df[col].clip(lower=0)
        if neg > 0:
            print(f'  {col}: исправлено {neg} отрицательных')

    # 2. Временные признаки
    df['Дата'] = pd.to_datetime(df['Дата'])
    df['ДеньНедели'] = df['Дата'].dt.dayofweek
    df['День'] = df['Дата'].dt.day
    df['IsWeekend'] = (df['ДеньНедели'] >= 5).astype(int)

    # 3. Агрегация
    group_cols = ['Дата', 'Пекарня', 'Номенклатура', 'Категория',
                  'Город', 'ДеньНедели', 'День', 'IsWeekend']
    agg = df.groupby(group_cols, as_index=False).agg(
        {'Продано': 'sum', 'Выпуск': 'sum', 'Остаток': 'sum'}
    )

    # 4. Feature Engineering
    agg = agg.sort_values(['Пекарня', 'Номенклатура', 'Дата']).reset_index(drop=True)
    grouped = agg.groupby(['Пекарня', 'Номенклатура'])

    n_days = agg['Дата'].nunique()
    lags = [1, 2, 3, 7]
    if n_days >= 21: lags.append(14)
    if n_days >= 37: lags.append(30)
    print(f'  Лаги: {lags} (дней в данных: {n_days})')

    for lag in lags:
        agg[f'sales_lag{lag}'] = grouped['Продано'].shift(lag)

    # Скользящие статистики
    agg['sales_roll_mean3'] = grouped['Продано'].transform(
        lambda x: x.shift(1).rolling(3, min_periods=1).mean())
    agg['sales_roll_mean7'] = grouped['Продано'].transform(
        lambda x: x.shift(1).rolling(7, min_periods=1).mean())
    agg['sales_roll_std7'] = grouped['Продано'].transform(
        lambda x: x.shift(1).rolling(7, min_periods=2).std())
    if n_days >= 21:
        agg['sales_roll_mean14'] = grouped['Продано'].transform(
            lambda x: x.shift(1).rolling(14, min_periods=7).mean())
    if n_days >= 37:
        agg['sales_roll_mean30'] = grouped['Продано'].transform(
            lambda x: x.shift(1).rolling(30, min_periods=14).mean())

    # Stock features
    agg['stock_lag1'] = grouped['Остаток'].shift(1)
    agg['stock_sales_ratio'] = agg['stock_lag1'] / (agg['sales_lag1'] + 1)
    agg['stock_deficit'] = (agg['sales_lag1'] - agg['stock_lag1']).clip(lower=0)

    # 5. Календарные признаки
    agg['Месяц'] = agg['Дата'].dt.month
    agg['НомерНедели'] = agg['Дата'].dt.isocalendar().week.astype(int)

    RU_HOLIDAYS = {
        (1,1),(1,2),(1,3),(1,4),(1,5),(1,6),(1,7),(1,8),
        (2,23),(3,8),(5,1),(5,9),(6,12),(11,4),
    }
    agg['is_holiday'] = agg['Дата'].apply(lambda d: int((d.month, d.day) in RU_HOLIDAYS))
    agg['is_pre_holiday'] = agg['Дата'].apply(
        lambda d: int(((d + pd.Timedelta(days=1)).month, (d + pd.Timedelta(days=1)).day) in RU_HOLIDAYS))
    agg['is_post_holiday'] = agg['Дата'].apply(
        lambda d: int(((d - pd.Timedelta(days=1)).month, (d - pd.Timedelta(days=1)).day) in RU_HOLIDAYS))
    agg['is_payday_week'] = agg['Дата'].dt.day.isin([4,5,6,19,20,21]).astype(int)

    # 6. Удаляем NaN
    before = len(agg)
    agg = agg.dropna().reset_index(drop=True)
    print(f'  Удалено {before - len(agg)} строк с NaN')
    print(f'  Итого: {agg.shape}')

    return agg


df = preprocess_data(FILE_PATH)
print(f'\nКолонки: {df.columns.tolist()}')
df.head()

## 2. Загрузка погоды (Open-Meteo API)

In [ ]:
CITIES = {
    'Казань':             {'lat': 55.7879, 'lon': 49.1233},
    'Чебоксары':          {'lat': 56.1439, 'lon': 47.2489},
    'Набережные Челны':   {'lat': 55.7439, 'lon': 52.4042},
    'Нижнекамск':         {'lat': 55.6386, 'lon': 51.8221},
    'Зеленодольск':       {'lat': 55.8430, 'lon': 48.5206},
    'Бугульма':           {'lat': 54.5393, 'lon': 52.7959},
    'Заинск':             {'lat': 55.2978, 'lon': 52.0046},
}


def fetch_weather(cities, start_date, end_date):
    cache_session = requests_cache.CachedSession('.weather_cache', expire_after=-1)
    retry_session = retry(cache_session, retries=3, backoff_factor=0.3)
    om = openmeteo_requests.Client(session=retry_session)

    daily_vars = [
        'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
        'precipitation_sum', 'rain_sum', 'snowfall_sum',
        'windspeed_10m_max', 'weathercode',
    ]

    frames = []
    for city, coords in cities.items():
        print(f'  {city}...', end=' ')
        try:
            resp = om.weather_api(
                'https://archive-api.open-meteo.com/v1/archive',
                params={
                    'latitude': coords['lat'], 'longitude': coords['lon'],
                    'start_date': start_date, 'end_date': end_date,
                    'daily': daily_vars, 'timezone': 'Europe/Moscow',
                }
            )[0].Daily()

            vals = {
                'temp_max': resp.Variables(0).ValuesAsNumpy(),
                'temp_min': resp.Variables(1).ValuesAsNumpy(),
                'temp_mean': resp.Variables(2).ValuesAsNumpy(),
                'precipitation': resp.Variables(3).ValuesAsNumpy(),
                'rain': resp.Variables(4).ValuesAsNumpy(),
                'snowfall': resp.Variables(5).ValuesAsNumpy(),
                'windspeed_max': resp.Variables(6).ValuesAsNumpy(),
                'weathercode': resp.Variables(7).ValuesAsNumpy(),
            }
            n = len(vals['temp_max'])
            dates = pd.date_range(
                start=pd.to_datetime(resp.Time(), unit='s', utc=True)
                        .tz_convert('Europe/Moscow').tz_localize(None),
                periods=n, freq='D'
            )
            cdf = pd.DataFrame({'Город': city, 'Дата': dates})
            for col, arr in vals.items():
                cdf[col] = arr[:n]
            frames.append(cdf)
            print('OK')
        except Exception as e:
            print(f'FAIL: {e}')

    weather = pd.concat(frames, ignore_index=True)
    weather['Дата'] = pd.to_datetime(weather['Дата']).dt.normalize()
    return weather


def enrich_weather(w):
    w['temp_range'] = w['temp_max'] - w['temp_min']
    w['is_rainy'] = (w['rain'] > 1.0).astype(int)
    w['is_snowy'] = (w['snowfall'] > 0.5).astype(int)
    w['is_cold'] = (w['temp_mean'] < 0).astype(int)
    w['is_warm'] = (w['temp_mean'] >= 15).astype(int)
    w['is_windy'] = (w['windspeed_max'] > 30).astype(int)
    w['is_bad_weather'] = ((w['is_rainy'] == 1) | (w['is_snowy'] == 1) | (w['is_windy'] == 1)).astype(int)

    def wmo_cat(code):
        c = int(code) if not np.isnan(code) else 0
        if c == 0: return 'clear'
        elif c <= 3: return 'cloudy'
        elif c <= 48: return 'fog'
        elif c <= 67: return 'rain'
        elif c <= 77: return 'snow'
        elif c <= 82: return 'showers'
        else: return 'storm'

    w['weather_category'] = w['weathercode'].apply(wmo_cat)
    return w


# Скачиваем погоду
start_date = df['Дата'].min().strftime('%Y-%m-%d')
end_date = df['Дата'].max().strftime('%Y-%m-%d')
print(f'Период: {start_date} -- {end_date}')

weather_df = fetch_weather(CITIES, start_date, end_date)
weather_df = enrich_weather(weather_df)
print(f'\nПогода: {weather_df.shape}')

## 3. Объединение данных

In [ ]:
# Календарные фичи (is_month_start, is_month_end)
df['is_month_start'] = (df['Дата'].dt.day <= 5).astype(int)
df['is_month_end'] = (df['Дата'].dt.day >= 25).astype(int)

# Мердж погоды
weather_df['Город'] = weather_df['Город'].astype(str)
df['Город'] = df['Город'].astype(str)

df_enriched = df.merge(
    weather_df.drop(columns=['weathercode']),
    on=['Дата', 'Город'], how='left'
)

# weather_category -> weather_cat_code
cat_map = {'clear': 0, 'cloudy': 1, 'fog': 2, 'rain': 3, 'showers': 4, 'snow': 5, 'storm': 6}
df_enriched['weather_cat_code'] = df_enriched['weather_category'].map(cat_map).fillna(0).astype(int)
df_enriched = df_enriched.drop(columns=['weather_category'])

print(f'Итоговый датасет: {df_enriched.shape}')
print(f'Nulls: {df_enriched.isnull().sum().sum()}')
print(f'\nКолонки: {df_enriched.columns.tolist()}')

## 4. Тюнинг LightGBM v6 (Optuna + GPU)

In [ ]:
FEATURES = [
    # Categorical
    'Пекарня', 'Номенклатура', 'Категория', 'Город',
    # Calendar (base)
    'ДеньНедели', 'День', 'IsWeekend', 'Месяц', 'НомерНедели',
    # Sales lags
    'sales_lag1', 'sales_lag2', 'sales_lag3', 'sales_lag7',
    'sales_lag14', 'sales_lag30',
    # Rolling stats
    'sales_roll_mean3', 'sales_roll_mean7', 'sales_roll_std7',
    'sales_roll_mean14', 'sales_roll_mean30',
    # Stock features
    'stock_lag1', 'stock_sales_ratio', 'stock_deficit',
    # Holiday / calendar
    'is_holiday', 'is_pre_holiday', 'is_post_holiday', 'is_payday_week',
    'is_month_start', 'is_month_end',
    # Weather (6 selected)
    'temp_mean', 'temp_range', 'precipitation',
    'is_cold', 'is_bad_weather', 'weather_cat_code',
]
TARGET = 'Продано'
N_TRIALS = 50

# Проверяем наличие признаков
available = [f for f in FEATURES if f in df_enriched.columns]
missing = [f for f in FEATURES if f not in df_enriched.columns]
if missing:
    print(f'Missing features: {missing}')
print(f'Using {len(available)} features')

In [ ]:
# Категориальные
for col in ['Пекарня', 'Номенклатура', 'Категория', 'Город', 'Месяц']:
    if col in df_enriched.columns:
        df_enriched[col] = df_enriched[col].astype('category')

# Train/test split (последние 2 дня = тест)
test_start = df_enriched['Дата'].max() - pd.Timedelta(days=2)
train = df_enriched[df_enriched['Дата'] < test_start]
test = df_enriched[df_enriched['Дата'] >= test_start]

X_tr, y_tr = train[available], train[TARGET]
X_te, y_te = test[available], test[TARGET]

print(f'Train: {X_tr.shape} ({train["Дата"].nunique()} days)')
print(f'Test:  {X_te.shape} ({test["Дата"].nunique()} days)')
print(f'Train period: {train["Дата"].min().date()} -- {train["Дата"].max().date()}')
print(f'Test period:  {test["Дата"].min().date()} -- {test["Дата"].max().date()}')

In [ ]:
def wmape(y_true, y_pred):
    return np.sum(np.abs(y_true - y_pred)) / (np.sum(y_true) + 1e-8) * 100


def objective(trial):
    params = {
        'n_estimators':      trial.suggest_int('n_estimators', 500, 3000),
        'learning_rate':     trial.suggest_float('learning_rate', 0.003, 0.2, log=True),
        'num_leaves':        trial.suggest_int('num_leaves', 16, 256),
        'max_depth':         trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample':         trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree':  trial.suggest_float('colsample_bytree', 0.4, 1.0),
        'reg_alpha':         trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda':        trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'random_state': 42, 'n_jobs': -1, 'verbose': -1,
    }
    if USE_GPU:
        params['device'] = 'gpu'

    m = LGBMRegressor(**params)
    m.fit(X_tr, y_tr)

    # Bias correction
    train_pred = np.maximum(m.predict(X_tr), 0)
    bias = np.mean(y_tr.values - train_pred)
    pred = np.maximum(m.predict(X_te) + bias, 0)

    return mean_absolute_error(y_te, pred)


# Baseline
print('=== Baseline (v5 params) ===')
bl_params = {
    'n_estimators': 1480, 'learning_rate': 0.00559902887165902,
    'num_leaves': 16, 'max_depth': 9, 'min_child_samples': 47,
    'subsample': 0.9738733982256087, 'colsample_bytree': 0.5250124486848033,
    'reg_alpha': 2.9277614297660098e-08, 'reg_lambda': 0.085417866288997,
    'random_state': 42, 'n_jobs': -1, 'verbose': -1,
}
if USE_GPU:
    bl_params['device'] = 'gpu'

bl = LGBMRegressor(**bl_params)
bl.fit(X_tr, y_tr)
bl_pred = np.maximum(bl.predict(X_te), 0)
bl_mae = mean_absolute_error(y_te, bl_pred)
bl_wm = wmape(y_te.values, bl_pred)
bl_bias = np.mean(y_te.values - bl_pred)
print(f'  MAE={bl_mae:.4f} | WMAPE={bl_wm:.2f}% | bias={bl_bias:+.3f}')

In [ ]:
%%time
print(f'Optuna: {N_TRIALS} trials...')
study = optuna.create_study(direction='minimize')

completed = [0]
def callback(study, trial):
    completed[0] += 1
    if completed[0] % 5 == 0 or completed[0] == 1:
        print(f'  [{completed[0]:>3}/{N_TRIALS}] MAE={trial.value:.4f} | best={study.best_value:.4f}')

study.optimize(objective, n_trials=N_TRIALS, callbacks=[callback])

print(f'\nBest MAE: {study.best_value:.4f}')
print('Best params:')
for k, v in study.best_params.items():
    print(f'  {k}: {v}')

## 5. Финальная модель + результаты

In [ ]:
# Финальная модель
best = study.best_params.copy()
best.update({'random_state': 42, 'n_jobs': -1, 'verbose': -1})
if USE_GPU:
    best['device'] = 'gpu'

model = LGBMRegressor(**best)
model.fit(X_tr, y_tr)

# Raw
pred_raw = np.maximum(model.predict(X_te), 0)
mae_raw = mean_absolute_error(y_te, pred_raw)
wm_raw = wmape(y_te.values, pred_raw)
bias_raw = np.mean(y_te.values - pred_raw)

# Bias corrected
train_pred = np.maximum(model.predict(X_tr), 0)
bias = np.mean(y_tr.values - train_pred)
pred = np.maximum(model.predict(X_te) + bias, 0)

mae = mean_absolute_error(y_te, pred)
rmse = np.sqrt(mean_squared_error(y_te, pred))
r2 = r2_score(y_te, pred)
wm = wmape(y_te.values, pred)
bias_after = np.mean(y_te.values - pred)

print('=' * 60)
print('  V6 FINAL RESULTS')
print('=' * 60)
print(f'  --- Raw (no bias correction) ---')
print(f'  MAE:   {mae_raw:.4f}')
print(f'  WMAPE: {wm_raw:.2f}%')
print(f'  Bias:  {bias_raw:+.4f}')
print()
print(f'  --- With bias correction ({bias:+.4f}) ---')
print(f'  MAE:   {mae:.4f}')
print(f'  RMSE:  {rmse:.4f}')
print(f'  R2:    {r2:.4f}')
print(f'  WMAPE: {wm:.2f}%')
print(f'  Bias:  {bias_after:+.4f}')
print('=' * 60)
print()
print('VERSION COMPARISON:')
print(f'  v3 (13 days):  MAE=3.7039')
print(f'  v5 (3 months): MAE=3.3900 | WMAPE=24.00%')
print(f'  v6 raw:        MAE={mae_raw:.4f} | WMAPE={wm_raw:.2f}%')
print(f'  v6 corrected:  MAE={mae:.4f} | WMAPE={wm:.2f}%')

In [ ]:
# Feature importance
imp = pd.DataFrame({
    'Feature': available,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print('Top-15 features:')
print(imp.head(15).to_string(index=False))

# Визуализация
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10, 8))
top20 = imp.head(20)
ax.barh(top20['Feature'][::-1], top20['Importance'][::-1])
ax.set_xlabel('Importance')
ax.set_title('V6 Model - Top 20 Feature Importance')
plt.tight_layout()
plt.show()

In [ ]:
# Сохраняем предсказания
results = X_te.copy()
results['fact'] = y_te.values
results['pred_corrected'] = pred
results['pred_raw'] = pred_raw
results['error'] = np.abs(pred - y_te.values)
results['bias_correction'] = bias
results.to_csv('v6_predictions.csv', index=False, encoding='utf-8-sig')
print('Predictions saved to v6_predictions.csv')